# Anggota 5 — Evaluasi Retrieval

Muhammad Faris El Hakim · 245150207111019

# 1. Peringkat Bukti dan Metrik

In [1]:
import json
from pathlib import Path
from statistics import mean

import pandas as pd
from IPython.display import display

with open("../Anggota3_Retrieval/output/hasil_pencarian.json", encoding="utf-8") as file:
    hasil_pencarian = json.load(file)
with open("../Anggota4/output/hasil_ekstraksi.json", encoding="utf-8") as file:
    hasil_ekstraksi = json.load(file)

hasil_soal = []
for soal in hasil_pencarian["hasil_per_pertanyaan"]:
    relevan = set(soal["passage_id"])
    peringkat = next(
        (rank for rank, pid in enumerate(soal["urutan_passage"], start=1) if pid in relevan),
        None,
    )
    top5 = soal["kandidat_top5"]
    ditemukan = sorted({p["passage_id"] for p in top5} & relevan)
    hasil_soal.append({
        "id_pertanyaan": soal["id_pertanyaan"],
        "penanya": soal["penanya"],
        "sumber_kontribusi": soal["sumber_kontribusi"],
        "pertanyaan": soal["pertanyaan"],
        "jawaban_rujukan": soal["jawaban"],
        "passage_id_relevan": soal["passage_id"],
        "bukti_rujukan": soal["bukti"],
        "peringkat_bukti_pertama": peringkat,
        "passage_relevan_di_top5": ditemukan,
        "precision_at_5": len(ditemukan) / 5,
        "recall_at_5": len(ditemukan) / len(relevan),
        "kandidat_top5": top5,
    })

jumlah_hit5 = sum(bool(baris["passage_relevan_di_top5"]) for baris in hasil_soal)
peringkat_bukti = [baris["peringkat_bukti_pertama"] for baris in hasil_soal]
ringkasan_retrieval = {
    "jumlah_pertanyaan": len(hasil_soal),
    "jumlah_passage": hasil_pencarian["jumlah_passage"],
    "hit_at_5": jumlah_hit5 / len(hasil_soal),
    "hit_at_5_jumlah": jumlah_hit5,
    "mrr": mean(1 / rank if rank is not None else 0 for rank in peringkat_bukti),
    "precision_at_5_macro": mean(baris["precision_at_5"] for baris in hasil_soal),
    "recall_at_5_macro": mean(baris["recall_at_5"] for baris in hasil_soal),
    "metode": "TF-IDF unigram dan cosine similarity dari Anggota 3",
    "catatan": "Peringkat dihitung pada kandidat dengan skor positif; urutan seri memakai passage_id.",
}
hasil_retrieval = {"ringkasan": ringkasan_retrieval, "hasil_per_pertanyaan": hasil_soal}

display(pd.DataFrame([ringkasan_retrieval])[
    ["jumlah_pertanyaan", "hit_at_5_jumlah", "hit_at_5", "mrr", "precision_at_5_macro", "recall_at_5_macro"]
].round(3))
pd.DataFrame(hasil_soal)[
    ["id_pertanyaan", "pertanyaan", "peringkat_bukti_pertama", "passage_relevan_di_top5"]
]

,jumlah_pertanyaan,hit_at_5_jumlah,hit_at_5,mrr,precision_at_5_macro,recall_at_5_macro
0,20,17,0.85,0.78,0.2,0.808


,id_pertanyaan,pertanyaan,peringkat_bukti_pertama,passage_relevan_di_top5
0,A2-001,Berapa sks minimal untuk menyelesaikan studi s...,3,[pa2020-II-I-7]
1,A2-002,Berapa lama waktu maksimal untuk Ujian Skripsi?,1,[pa2020-VI-D-10]
2,A2-003,Apa itu skripsi?,19,[]
3,A2-004,Apa nilai huruf untuk nilai angka 78?,6,[]
4,A3-001,Berapa lama waktu maksimal untuk menyelesaikan...,2,[pa2020-II-I-8]
5,A3-002,Kapan batas pengajuan cuti akademik setelah da...,1,[pa2020-III-G-2]
6,A3-003,Apa rumus indeks prestasi semester?,25,[]
7,A4-001,Berapa persentase kehadiran minimal agar mahas...,1,"[pa2020-II-F-9, pa2020-III-K-1]"
8,A4-002,Berapa menit batas maksimal keterlambatan meng...,1,[tatatertib-X-S1-3]
9,A4-003,Berapa bobot nilai huruf B+?,1,"[pa2020-bobot-2, penilaian-bobot-2]"


Hit@5 menghitung soal dengan bukti relevan dalam lima besar. MRR menghitung rata-rata kebalikan peringkat bukti pertama, termasuk bukti yang muncul setelah peringkat lima.

# 2. Jawaban Bot

In [2]:
id_contoh = "FARIS-003"
peringkat_contoh = pd.DataFrame(hasil_soal).loc[:, ["id_pertanyaan", "peringkat_bukti_pertama"]]
contoh = pd.DataFrame(hasil_ekstraksi).merge(peringkat_contoh, on="id_pertanyaan", validate="one_to_one")
contoh.loc[contoh["id_pertanyaan"] == id_contoh, [
    "pertanyaan", "jawaban", "jawaban_rujukan", "peringkat_bukti_pertama", "passage_id", "sumber"
]]

,pertanyaan,jawaban,jawaban_rujukan,peringkat_bukti_pertama,passage_id,sumber
13,Berapa sks minimal yang harus dikumpulkan untu...,120 sks,138 sks,1,pa2020-VI-A-3,Buku Pedoman Akademik Fakultas Ilmu Komputer U...


Untuk FARIS-003, passage rujukan **138 SKS** berada di peringkat pertama. Sistem memilih **120 SKS** dari aturan tentang syarat membuat skripsi. Ini menunjukkan perbedaan antara menemukan bukti dan memilih jawaban.

# 3. Simpan Metrik dan Bukti

In [3]:
Path("output").mkdir(exist_ok=True)
with open("output/hasil_retrieval.json", "w", encoding="utf-8") as file:
    json.dump(hasil_retrieval, file, ensure_ascii=False, indent=2)
    file.write("\n")

Hasil disimpan per ID soal, termasuk jawaban rujukan, ID bukti, dan peringkat bukti pertama.